## EC2 Credentials and IAM Roles

Welcome to the final lesson of Developing with Core AWS Services. In this lesson, you will learn how AWS credentials work on EC2 instances and how IAM roles make it possible for your code to securely access AWS services. This ties together everything you've learned so far — from the boto3 Universal Pattern to working with S3, DynamoDB, SQS, and SNS — by revealing how your code automatically and securely obtains its credentials when running on AWS.

When you launch an EC2 instance, you often want it to interact with other AWS services, such as S3, DynamoDB, or Lambda. To do this, the instance needs credentials — just like you need a username and password to log in to a website. However, hardcoding credentials (putting them directly in your code) is risky and not recommended. If someone gets access to your code, they could also get your credentials and misuse your AWS account.

In this lesson, you will see how AWS provides a secure and automatic way for EC2 instances to get credentials using IAM roles and the EC2 metadata service.

---

## Quick Recall: AWS Credentials and SDKs

Before we dive in, let's quickly remind ourselves what AWS credentials are and how SDKs use them.

AWS credentials are a pair of keys (an access key and a secret key) that allow you to make API calls to AWS services. In previous lessons, you have used these credentials with the AWS SDK for Python, called boto3. The SDK uses these credentials to sign requests and prove your identity to AWS.

For example, you might have seen code like this:

```python
import boto3

# Create an S3 client using credentials from your environment
s3 = boto3.client('s3')
```

Here, boto3 automatically looks for credentials in several places, such as environment variables, configuration files, or the EC2 metadata service (which we'll cover next).

Hardcoding credentials in your code is not safe. Instead, AWS recommends using IAM roles, especially for EC2 instances.

---

## How EC2 Gets Credentials: The Metadata Service

When you launch an EC2 instance, AWS provides a special service called the metadata service. This service is only accessible from inside the instance. It provides information about the instance, such as its ID, type, and, most importantly, temporary credentials if an IAM role is attached.

The metadata service is available at a special IP address: `http://169.254.169.254/latest/meta-data/`

You can use simple tools like `curl` to access this information from within the instance. For example:

```shell
curl http://169.254.169.254/latest/meta-data/instance-id
```

This command returns the instance ID, such as:

```text
i-0123456789abcdef0
```

The metadata service provides much more than just the instance ID. You can explore various endpoints to get information about the instance's AMI, security groups, network interfaces, and more. Try running these commands directly on an EC2 instance to see the full range of available metadata:

```shell
# List all available metadata categories
curl http://169.254.169.254/latest/meta-data/

# Get specific information
curl http://169.254.169.254/latest/meta-data/ami-id
curl http://169.254.169.254/latest/meta-data/instance-type
curl http://169.254.169.254/latest/meta-data/placement/region
```

Running these commands directly will give you a complete view of what AWS-related information is accessible from within your instance.

If your instance has an IAM role attached, you can also get temporary credentials from the metadata service. These credentials are rotated automatically and are only valid for a short time, making them much safer than hardcoded keys.

---

## IAM Roles for EC2: Secure Access in Action

An IAM role is a set of permissions that you can assign to AWS resources. When you attach an IAM role to an EC2 instance, AWS automatically provides temporary credentials to that instance through the metadata service.

This means your code running on the instance can access AWS services securely, without you ever needing to put credentials in your code.

For example, if you attach a role that allows access to S3, any code running on the instance can use boto3 to interact with S3, and the SDK will automatically use the credentials provided by the metadata service.

This is how AWS keeps your credentials safe and makes development easier.

---

## Code Walkthrough: Discovering Credentials on EC2

Let's walk through a simple example that shows how an EC2 instance can access its own metadata and check for IAM role credentials. We'll build this step by step.

### 1. Writing a User Data Script

When you launch an EC2 instance, you can provide a user data script. This script runs automatically when the instance starts. Here's a simple Bash script that prints out some instance information and checks for an IAM role:

```shell
#!/bin/bash

echo "============ EC2 DEMO START ============"
echo "✅ User data script executing on EC2!"
echo "🔍 Instance Information:"

# Get metadata using curl
echo "   Instance ID: $(curl -s http://169.254.169.254/latest/meta-data/instance-id)"
echo "   AMI ID: $(curl -s http://169.254.169.254/latest/meta-data/ami-id)"  
echo "   Instance Type: $(curl -s http://169.254.169.254/latest/meta-data/instance-type)"
echo "   Region: $(curl -s http://169.254.169.254/latest/meta-data/placement/region)"

echo "🔍 IAM Role Check:"
if curl -s -f http://169.254.169.254/latest/meta-data/iam/security-credentials/ > /dev/null; then
    ROLE=$(curl -s http://169.254.169.254/latest/meta-data/iam/security-credentials/)
    echo "   ✅ IAM Role: $ROLE"
else
    echo "   ❌ No IAM role attached"
fi

echo "💡 This demonstrates EC2 metadata service access!"
echo "============ EC2 DEMO END ============"
```

**Explanation:**

* The script uses `curl` to fetch information from the metadata service.
* It prints the instance ID, AMI ID, instance type, and region.
* It checks if an IAM role is attached by trying to access the `iam/security-credentials/` endpoint.
* If a role is found, it prints the role name; otherwise, it says no role is attached.

**Sample Output:**

```text
============ EC2 DEMO START ============
✅ User data script executing on EC2!
🔍 Instance Information:
   Instance ID: i-0123456789abcdef0
   AMI ID: ami-0c02fb55956c7d316
   Instance Type: t3.micro
   Region: us-west-2
🔍 IAM Role Check:
   ✅ IAM Role: MyEC2Role
💡 This demonstrates EC2 metadata service access!
============ EC2 DEMO END ============
```

### 2. Launching an EC2 Instance with the Script

Now, let's see how you can launch an EC2 instance and provide this script as user data using Python and boto3.

First, you need to load the script from a file:

```python
def load_user_data_script():
    """Load the bash script for EC2 user data"""
    try:
        with open('ec2_demo.sh', 'r') as f:
            return f.read()
    except FileNotFoundError:
        print("❌ Error: ec2_demo.sh file not found")
        return None
```

**Explanation:**

* This function reads the contents of `ec2_demo.sh` (the script above) so it can be passed to the EC2 instance as user data.

Next, you need a function to get the latest Amazon Linux 2 AMI ID:

```python
def get_latest_amazon_linux_ami():
    """Get the latest Amazon Linux 2 AMI ID for the current region"""
    ssm = boto3.client('ssm')
    try:
        response = ssm.get_parameter(
            Name='/aws/service/ami-amazon-linux-latest/amzn2-ami-hvm-x86_64-gp2'
        )
        return response['Parameter']['Value']
    except Exception as e:
        print(f"❌ Error getting latest AMI: {e}")
        return None
```

**Explanation:**

* This function uses the AWS Systems Manager Parameter Store to get the latest Amazon Linux 2 AMI ID.
* AWS automatically maintains these parameters with the most current AMI IDs for each region.
* This ensures you're always using an up-to-date AMI without hardcoding specific IDs.

Now you can launch the EC2 instance:

```python
import boto3

def launch_ec2_demo():
    """Launch EC2 instance with simple bash demo"""
    print("🚀 Launching EC2 instance...")
    
    user_data = load_user_data_script()
    if not user_data:
        return None
    
    ami_id = get_latest_amazon_linux_ami()
    if not ami_id:
        return None
    
    print(f"📋 Using AMI: {ami_id}")
    
    ec2 = boto3.client('ec2')
    
    try:
        response = ec2.run_instances(
            ImageId=ami_id,
            MinCount=1,
            MaxCount=1,
            InstanceType='t3.micro',
            UserData=user_data,
            TagSpecifications=[
                {
                    'ResourceType': 'instance',
                    'Tags': [{'Key': 'Name', 'Value': 'metadata-demo'}]
                }
            ]
        )
        
        instance_id = response['Instances'][0]['InstanceId']
        print(f"✅ Launched instance: {instance_id}")
        return instance_id
        
    except Exception as e:
        print(f"❌ Error launching instance: {e}")
        return None
```

**Explanation:**

* This function uses boto3 to launch a new EC2 instance.
* It first gets the latest Amazon Linux 2 AMI ID dynamically, ensuring compatibility with the current region.
* The `UserData` parameter is set to the contents of your script.
* The instance will run the script on startup, printing out metadata and IAM role information.

### 3. Retrieving the Output

After the instance starts, you can retrieve the output of the user data script by checking the instance's console output:

```python
import time

def wait_for_console_output(instance_id):
    """Wait for console output to appear and parse it"""
    print(f"\n⏳ Waiting for console output from {instance_id}...")
    print("💡 This can take 5-10 minutes - please be patient!")
    
    ec2 = boto3.client('ec2')
    
    # Wait for running state first
    waiter = ec2.get_waiter('instance_running')
    waiter.wait(InstanceIds=[instance_id])
    print("✅ Instance is running")
    
    # Wait for console output
    for attempt in range(15):  # Try for up to 15 minutes
        time.sleep(60)  # Wait 1 minute between attempts
        response = ec2.get_console_output(InstanceId=instance_id)
        console_output = response.get('Output', '')
        if console_output and 'EC2 DEMO START' in console_output:
            print("\n📋 Actual Demo Results from EC2:")
            print("=" * 60)
            print(console_output)
            print("=" * 60)
            return True
        else:
            print("   📭 No console output yet")
    print("⏰ Timeout waiting for demo output")
    return False
```

**Explanation:**

* This function waits for the instance to start and then checks the console output for the results of your script.
* It looks for the markers `EC2 DEMO START` and `EC2 DEMO END` to find your script's output.

**Sample Output:**

```text
============ EC2 DEMO START ============
✅ User data script executing on EC2!
🔍 Instance Information:
   Instance ID: i-0123456789abcdef0
   AMI ID: ami-0c02fb55956c7d316
   Instance Type: t3.micro
   Region: us-west-2
🔍 IAM Role Check:
   ❌ No IAM role attached
💡 This demonstrates EC2 metadata service access!
============ EC2 DEMO END ============
```

If you attach an IAM role, you will see the role name instead of the "No IAM role attached" message.

---

## Summary and What's Next

In this lesson, you learned:

* Why EC2 instances need credentials to access AWS services.
* The risks of hardcoding credentials and why it's not recommended.
* How the EC2 metadata service provides instance information and, if available, IAM role credentials.
* How IAM roles allow EC2 instances to securely and automatically get temporary credentials.
* How to use a user data script and Python code to discover and display these credentials on a real EC2 instance.

You are now ready to practice these concepts in hands-on exercises. In the next section, you'll get to try launching EC2 instances, attaching IAM roles, and exploring the metadata service yourself. This will help you build a strong foundation for working securely and efficiently with AWS services from your EC2 instances.

## Adding EC2 Metadata Service Information

Now that you've seen how the metadata service provides instance information such as ID, AMI, type, and region, let's practice adding one more piece of data to make the picture complete.

Your task is to modify the `ec2_demo.sh` script by adding availability zone information to the instance details. Look at the existing pattern in the script — you'll see how other metadata is retrieved using curl commands with specific endpoints.

Add a new line in the "Instance Information" section, right after the region line, that retrieves and displays the availability zone. Use the same formatting style as the other lines, and follow the endpoint pattern you see in the existing code.

Once you complete this change, you can run the Python script to launch your EC2 instance and see your new availability zone information appear alongside the other instance details in the console output.

⚠️ **Important:** Because launching and initializing a real EC2 instance takes 5-10 minutes, clicking the "Run" button in this environment will not execute the script (it would simply time out). To see your changes in action, you must manually run the following command in the terminal:

```shell
python main.py
```

This allows you to monitor the progress in real-time as the instance boots up and executes your script.

**ec2_demo.sh**

```shell
#!/bin/bash

echo "============ EC2 DEMO START ============"
echo "✅ User data script executing on EC2!"
echo "🔍 Instance Information:"

# Get metadata using curl
echo "   Instance ID: $(curl -s http://169.254.169.254/latest/meta-data/instance-id)"
echo "   AMI ID: $(curl -s http://169.254.169.254/latest/meta-data/ami-id)"  
echo "   Instance Type: $(curl -s http://169.254.169.254/latest/meta-data/instance-type)"
echo "   Region: $(curl -s http://169.254.169.254/latest/meta-data/placement/region)"
# TODO: Add availability zone information here using the metadata service

echo "🔍 IAM Role Check:"
if curl -s -f http://169.254.169.254/latest/meta-data/iam/security-credentials/ > /dev/null; then
    ROLE=$(curl -s http://169.254.169.254/latest/meta-data/iam/security-credentials/)
    echo "   ✅ IAM Role: $ROLE"
else
    echo "   ❌ No IAM role attached"
fi

echo "💡 This demonstrates EC2 metadata service access!"
echo "============ EC2 DEMO END ============"
```

**main.py**

```python
import boto3
import time
from botocore.exceptions import ClientError

def load_user_data_script():
    """Load the bash script for EC2 user data"""
    try:
        with open('ec2_demo.sh', 'r') as f:
            return f.read()
    except FileNotFoundError:
        print("❌ Error: ec2_demo.sh file not found")
        return None

def get_latest_amazon_linux_ami():
    """Get the latest Amazon Linux 2 AMI ID for the current region"""
    ssm = boto3.client('ssm')
    try:
        response = ssm.get_parameter(
            Name='/aws/service/ami-amazon-linux-latest/amzn2-ami-hvm-x86_64-gp2'
        )
        return response['Parameter']['Value']
    except Exception as e:
        print(f"❌ Error getting latest AMI: {e}")
        return None

def launch_ec2_demo():
    """Launch EC2 instance with simple bash demo"""
    print("🚀 Launching EC2 instance...")
    
    user_data = load_user_data_script()
    if not user_data:
        return None
    
    ec2 = boto3.client('ec2')
    
    try:
        response = ec2.run_instances(
            ImageId=get_latest_amazon_linux_ami(),
            MinCount=1,
            MaxCount=1,
            InstanceType='t3.micro',
            UserData=user_data,
            TagSpecifications=[
                {
                    'ResourceType': 'instance',
                    'Tags': [{'Key': 'Name', 'Value': 'metadata-demo'}]
                }
            ]
        )
        
        instance_id = response['Instances'][0]['InstanceId']
        print(f"✅ Launched instance: {instance_id}")
        return instance_id
        
    except Exception as e:
        print(f"❌ Error launching instance: {e}")
        return None

def wait_for_console_output(instance_id):
    """Wait patiently for console output to appear and parse it correctly"""
    print(f"\n⏳ Waiting for console output from {instance_id}...")
    print("💡 This can take 5-10 minutes - please be patient!")
    
    ec2 = boto3.client('ec2')
    
    # Wait for running state first
    waiter = ec2.get_waiter('instance_running')
    waiter.wait(InstanceIds=[instance_id])
    print("✅ Instance is running")
    
    # Wait longer for console output - up to 15 minutes
    for attempt in range(15):  # Try for up to 15 minutes
        wait_time = 60  # 1 minute intervals
        print(f"⏳ Checking for output (attempt {attempt + 1}/15) - waiting {wait_time}s...")
        time.sleep(wait_time)
        
        try:
            response = ec2.get_console_output(InstanceId=instance_id)
            console_output = response.get('Output', '')
            
            if console_output and 'EC2 DEMO START' in console_output:
                print("\n📋 Actual Demo Results from EC2:")
                print("=" * 60)
                
                # Parse cloud-init logs to extract demo output
                lines = console_output.split('\n')
                demo_lines = []
                in_demo = False
                
                for line in lines:
                    if 'EC2 DEMO START' in line:
                        in_demo = True
                        # Extract message from cloud-init log
                        if 'cloud-init[' in line and ']: ' in line:
                            msg = line.split(']: ', 1)[1]
                            demo_lines.append(msg)
                    elif 'EC2 DEMO END' in line:
                        if 'cloud-init[' in line and ']: ' in line:
                            msg = line.split(']: ', 1)[1]
                            demo_lines.append(msg)
                        break
                    elif in_demo and 'cloud-init[' in line and ']: ' in line:
                        # Extract just the message part after ]: 
                        msg = line.split(']: ', 1)[1]
                        demo_lines.append(msg)
                
                # Display the extracted demo output
                if demo_lines:
                    for demo_line in demo_lines:
                        print(demo_line)
                else:
                    print("Demo markers found but content extraction failed")
                    print("Raw demo section:")
                    for line in lines:
                        if 'DEMO' in line:
                            print(line)
                            
                print("=" * 60)
                return True
                
            elif console_output:
                char_count = len(console_output)
                print(f"   📄 Console output available ({char_count} chars) but demo not found")
                
                # Show some context on later attempts
                if attempt >= 8:
                    print("   Showing lines with 'cloud-init' for debugging:")
                    lines = console_output.split('\n')
                    cloud_init_lines = [line for line in lines if 'cloud-init[' in line]
                    for line in cloud_init_lines[-10:]:  # Last 10 cloud-init lines
                        print(f"   {line}")
            else:
                print("   📭 No console output yet")
                
        except Exception as e:
            print(f"   ❌ Error checking output: {e}")
    
    print("⏰ Timeout waiting for demo output")
    print(f"💡 You can check manually with:")
    print(f"   aws ec2 get-console-output --instance-id {instance_id} --query 'Output' --output text | grep -A 10 -B 2 DEMO")
    return False

def cleanup(instance_id):
    """Terminate the demo instance"""
    if not instance_id:
        return
        
    terminate = input(f"\n🗑️ Terminate instance {instance_id}? (y/N): ").strip().lower()
    if terminate == 'y':
        try:
            ec2 = boto3.client('ec2')
            ec2.terminate_instances(InstanceIds=[instance_id])
            print("✅ Instance terminated")
        except Exception as e:
            print(f"❌ Error: {e}")
    else:
        print(f"⚠️ Instance {instance_id} still running")

def main():
    print("🚀 EC2 IAM Role Credential Discovery Demo")
    print()
    print("💡 This demo shows REAL execution from EC2:")
    print("   • How EC2 instances access the metadata service")
    print("   • Where IAM role credentials are provided") 
    print("   • How boto3 would discover credentials automatically")
    print()
    
    instance_id = launch_ec2_demo()
    
    if instance_id:
        success = wait_for_console_output(instance_id)
        
        if success:
            print("\n🎯 What this demonstrated:")
            print("   ✅ User data script executed successfully on EC2")
            print("   ✅ EC2 metadata service is accessible from instances")
            print("   ✅ Shows actual instance information (ID, AMI, type, region)")
            print("   ✅ Demonstrates where IAM role credentials would appear")
            print("   💡 Without IAM role: No AWS API access")
            print("   💡 With IAM role: boto3 automatically gets credentials from metadata service")
        else:
            print("\n💡 The demo ran, but console output timing varies")
            print("   You can check the results manually with the command shown above")
        
        print("\n🔑 Key Learning Points:")
        print("   • EC2 metadata service URL: http://169.254.169.254/latest/meta-data/")
        print("   • IAM role credentials URL: .../iam/security-credentials/RoleName")  
        print("   • boto3 automatically discovers and uses these credentials")
        print("   • No IAM role = NoCredentialsError when calling AWS APIs")
        
        cleanup(instance_id)

if __name__ == "__main__":
    main()
```

Here is the completed `ec2_demo.sh` with the availability zone line added right after the region line:

```shell
#!/bin/bash

echo "============ EC2 DEMO START ============"
echo "✅ User data script executing on EC2!"
echo "🔍 Instance Information:"

# Get metadata using curl
echo "   Instance ID: $(curl -s http://169.254.169.254/latest/meta-data/instance-id)"
echo "   AMI ID: $(curl -s http://169.254.169.254/latest/meta-data/ami-id)"  
echo "   Instance Type: $(curl -s http://169.254.169.254/latest/meta-data/instance-type)"
echo "   Region: $(curl -s http://169.254.169.254/latest/meta-data/placement/region)"
echo "   Availability Zone: $(curl -s http://169.254.169.254/latest/meta-data/placement/availability-zone)"

echo "🔍 IAM Role Check:"
if curl -s -f http://169.254.169.254/latest/meta-data/iam/security-credentials/ > /dev/null; then
    ROLE=$(curl -s http://169.254.169.254/latest/meta-data/iam/security-credentials/)
    echo "   ✅ IAM Role: $ROLE"
else
    echo "   ❌ No IAM role attached"
fi

echo "💡 This demonstrates EC2 metadata service access!"
echo "============ EC2 DEMO END ============"
```

`main.py` stays unchanged — it just loads `ec2_demo.sh` as-is and launches the instance.

## Customizing EC2 Launch Parameters

Perfect! You've learned how the metadata service works and have practiced retrieving instance information. Now, let's take the next step by customizing how you launch EC2 instances with specific configuration parameters.

Your objective is to modify two key settings in the `launch_ec2_demo()` function within `main.py`. You need to:

* Change the instance type from `t3.micro` to `t2.micro`
* Update the instance name tag from `metadata-demo` to `my-demo-instance`

Look for the `run_instances` call in the function — you'll find the `InstanceType` parameter and the `TagSpecifications` section where these changes need to be made. The TODO comments will guide you to the exact lines.

These simple modifications will teach you how boto3 parameters directly control your AWS resources, giving you the foundation for customizing instance launches in real projects.

Once you complete this change, you can run the Python script to launch your EC2 instance and see your new availability zone information appear alongside the other instance details in the console output.

⚠️ **Important:** Because launching and initializing a real EC2 instance takes 5-10 minutes, clicking the "Run" button in this environment will not execute the script (it would simply time out). To see your changes in action, you must manually run the following command in the terminal:

```shell
python main.py
```

This allows you to monitor the progress in real-time as the instance boots up and executes your script.

**main.py**

```python
import boto3
import time
from botocore.exceptions import ClientError

def load_user_data_script():
    """Load the bash script for EC2 user data"""
    try:
        with open('ec2_demo.sh', 'r') as f:
            return f.read()
    except FileNotFoundError:
        print("❌ Error: ec2_demo.sh file not found")
        return None

def get_latest_amazon_linux_ami():
    """Get the latest Amazon Linux 2 AMI ID for the current region"""
    ssm = boto3.client('ssm')
    try:
        response = ssm.get_parameter(
            Name='/aws/service/ami-amazon-linux-latest/amzn2-ami-hvm-x86_64-gp2'
        )
        return response['Parameter']['Value']
    except Exception as e:
        print(f"❌ Error getting latest AMI: {e}")
        return None

def launch_ec2_demo():
    """Launch EC2 instance with simple bash demo"""
    print("🚀 Launching EC2 instance...")
    
    user_data = load_user_data_script()
    if not user_data:
        return None
    
    ec2 = boto3.client('ec2')
    
    try:
        response = ec2.run_instances(
            ImageId=get_latest_amazon_linux_ami(),
            MinCount=1,
            MaxCount=1,
            # TODO: Change the instance type from 't3.micro' to 't2.micro'
            InstanceType='t3.micro',
            UserData=user_data,
            TagSpecifications=[
                {
                    'ResourceType': 'instance',
                    # TODO: Change the tag value from 'metadata-demo' to 'my-demo-instance'
                    'Tags': [{'Key': 'Name', 'Value': 'metadata-demo'}]
                }
            ]
        )
        
        instance_id = response['Instances'][0]['InstanceId']
        print(f"✅ Launched instance: {instance_id}")
        return instance_id
        
    except Exception as e:
        print(f"❌ Error launching instance: {e}")
        return None

def wait_for_console_output(instance_id):
    """Wait patiently for console output to appear and parse it correctly"""
    print(f"\n⏳ Waiting for console output from {instance_id}...")
    print("💡 This can take 5-10 minutes - please be patient!")
    
    ec2 = boto3.client('ec2')
    
    # Wait for running state first
    waiter = ec2.get_waiter('instance_running')
    waiter.wait(InstanceIds=[instance_id])
    print("✅ Instance is running")
    
    # Wait longer for console output - up to 15 minutes
    for attempt in range(15):  # Try for up to 15 minutes
        wait_time = 60  # 1 minute intervals
        print(f"⏳ Checking for output (attempt {attempt + 1}/15) - waiting {wait_time}s...")
        time.sleep(wait_time)
        
        try:
            response = ec2.get_console_output(InstanceId=instance_id)
            console_output = response.get('Output', '')
            
            if console_output and 'EC2 DEMO START' in console_output:
                print("\n📋 Actual Demo Results from EC2:")
                print("=" * 60)
                
                # Parse cloud-init logs to extract demo output
                lines = console_output.split('\n')
                demo_lines = []
                in_demo = False
                
                for line in lines:
                    if 'EC2 DEMO START' in line:
                        in_demo = True
                        # Extract message from cloud-init log
                        if 'cloud-init[' in line and ']: ' in line:
                            msg = line.split(']: ', 1)[1]
                            demo_lines.append(msg)
                    elif 'EC2 DEMO END' in line:
                        if 'cloud-init[' in line and ']: ' in line:
                            msg = line.split(']: ', 1)[1]
                            demo_lines.append(msg)
                        break
                    elif in_demo and 'cloud-init[' in line and ']: ' in line:
                        # Extract just the message part after ]: 
                        msg = line.split(']: ', 1)[1]
                        demo_lines.append(msg)
                
                # Display the extracted demo output
                if demo_lines:
                    for demo_line in demo_lines:
                        print(demo_line)
                else:
                    print("Demo markers found but content extraction failed")
                    print("Raw demo section:")
                    for line in lines:
                        if 'DEMO' in line:
                            print(line)
                            
                print("=" * 60)
                return True
                
            elif console_output:
                char_count = len(console_output)
                print(f"   📄 Console output available ({char_count} chars) but demo not found")
                
                # Show some context on later attempts
                if attempt >= 8:
                    print("   Showing lines with 'cloud-init' for debugging:")
                    lines = console_output.split('\n')
                    cloud_init_lines = [line for line in lines if 'cloud-init[' in line]
                    for line in cloud_init_lines[-10:]:  # Last 10 cloud-init lines
                        print(f"   {line}")
            else:
                print("   📭 No console output yet")
                
        except Exception as e:
            print(f"   ❌ Error checking output: {e}")
    
    print("⏰ Timeout waiting for demo output")
    print(f"💡 You can check manually with:")
    print(f"   aws ec2 get-console-output --instance-id {instance_id} --query 'Output' --output text | grep -A 10 -B 2 DEMO")
    return False

def cleanup(instance_id):
    """Terminate the demo instance"""
    if not instance_id:
        return
        
    terminate = input(f"\n🗑️ Terminate instance {instance_id}? (y/N): ").strip().lower()
    if terminate == 'y':
        try:
            ec2 = boto3.client('ec2')
            ec2.terminate_instances(InstanceIds=[instance_id])
            print("✅ Instance terminated")
        except Exception as e:
            print(f"❌ Error: {e}")
    else:
        print(f"⚠️ Instance {instance_id} still running")

def main():
    print("🚀 EC2 IAM Role Credential Discovery Demo")
    print()
    print("💡 This demo shows REAL execution from EC2:")
    print("   • How EC2 instances access the metadata service")
    print("   • Where IAM role credentials are provided") 
    print("   • How boto3 would discover credentials automatically")
    print()
    
    instance_id = launch_ec2_demo()
    
    if instance_id:
        success = wait_for_console_output(instance_id)
        
        if success:
            print("\n🎯 What this demonstrated:")
            print("   ✅ User data script executed successfully on EC2")
            print("   ✅ EC2 metadata service is accessible from instances")
            print("   ✅ Shows actual instance information (ID, AMI, type, region)")
            print("   ✅ Demonstrates where IAM role credentials would appear")
            print("   💡 Without IAM role: No AWS API access")
            print("   💡 With IAM role: boto3 automatically gets credentials from metadata service")
        else:
            print("\n💡 The demo ran, but console output timing varies")
            print("   You can check the results manually with the command shown above")
        
        print("\n🔑 Key Learning Points:")
        print("   • EC2 metadata service URL: http://169.254.169.254/latest/meta-data/")
        print("   • IAM role credentials URL: .../iam/security-credentials/RoleName")  
        print("   • boto3 automatically discovers and uses these credentials")
        print("   • No IAM role = NoCredentialsError when calling AWS APIs")
        
        cleanup(instance_id)

if __name__ == "__main__":
    main()
```

**ec2_demo.sh**

```shell
#!/bin/bash

echo "============ EC2 DEMO START ============"
echo "✅ User data script executing on EC2!"
echo "🔍 Instance Information:"

# Get metadata using curl
echo "   Instance ID: $(curl -s http://169.254.169.254/latest/meta-data/instance-id)"
echo "   AMI ID: $(curl -s http://169.254.169.254/latest/meta-data/ami-id)"  
echo "   Instance Type: $(curl -s http://169.254.169.254/latest/meta-data/instance-type)"
echo "   Region: $(curl -s http://169.254.169.254/latest/meta-data/placement/region)"

echo "🔍 IAM Role Check:"
if curl -s -f http://169.254.169.254/latest/meta-data/iam/security-credentials/ > /dev/null; then
    ROLE=$(curl -s http://169.254.169.254/latest/meta-data/iam/security-credentials/)
    echo "   ✅ IAM Role: $ROLE"
else
    echo "   ❌ No IAM role attached"
fi

echo "💡 This demonstrates EC2 metadata service access!"
echo "============ EC2 DEMO END ============"
```

Here is the completed `launch_ec2_demo()` with the instance type and tag value fixed:

```python
def launch_ec2_demo():
    """Launch EC2 instance with simple bash demo"""
    print("🚀 Launching EC2 instance...")
    
    user_data = load_user_data_script()
    if not user_data:
        return None
    
    ec2 = boto3.client('ec2')
    
    try:
        response = ec2.run_instances(
            ImageId=get_latest_amazon_linux_ami(),
            MinCount=1,
            MaxCount=1,
            InstanceType='t2.micro',
            UserData=user_data,
            TagSpecifications=[
                {
                    'ResourceType': 'instance',
                    'Tags': [{'Key': 'Name', 'Value': 'my-demo-instance'}]
                }
            ]
        )
        
        instance_id = response['Instances'][0]['InstanceId']
        print(f"✅ Launched instance: {instance_id}")
        return instance_id
        
    except Exception as e:
        print(f"❌ Error launching instance: {e}")
        return None
```

The rest of `main.py` and `ec2_demo.sh` stay unchanged.

## Discovering Temporary Credentials in Action

Excellent! You've successfully learned how to detect IAM roles and customize instance launch parameters. Now, let's dive deeper into the credential discovery process by exploring the actual temporary credentials that AWS provides.

Your task is to enhance the IAM role check section in `ec2_demo.sh` to retrieve and display the actual temporary credentials when available. This will show you exactly where boto3 automatically finds credentials.

Inside the existing IAM role check block, you need to add code that:

* Fetches the complete credential information from the metadata service using the role name
* Extracts the `AccessKeyId` and `SecretAccessKey` from the JSON response
* Displays the first 20 characters of each credential (for security reasons)
* Shows when these temporary credentials expire

Look for the TODO comments in the script — they will guide you through adding the credential retrieval logic right after the role name is displayed. You'll be working with the same metadata endpoint pattern you've already seen, but accessing the specific role's credential data.

The `main.py` script will automatically create an IAM role and attach it to your EC2 instance, so you can see real temporary credentials in action. This exercise demonstrates the exact same process that boto3 uses automatically when your code runs on EC2 with an IAM role attached.

Once you complete this change, you can run the Python script to launch your EC2 instance and see your new availability zone information appear alongside the other instance details in the console output.

⚠️ **Important:** Because launching and initializing a real EC2 instance takes 5-10 minutes, clicking the "Run" button in this environment will not execute the script (it would simply time out). To see your changes in action, you must manually run the following command in the terminal:

```shell
python main.py
```

**ec2_demo.sh**

```shell
#!/bin/bash

echo "============ EC2 DEMO START ============"
echo "✅ User data script executing on EC2!"
echo "🔍 Instance Information:"

# Get metadata using curl
echo "   Instance ID: $(curl -s http://169.254.169.254/latest/meta-data/instance-id)"
echo "   AMI ID: $(curl -s http://169.254.169.254/latest/meta-data/ami-id)"  
echo "   Instance Type: $(curl -s http://169.254.169.254/latest/meta-data/instance-type)"
echo "   Region: $(curl -s http://169.254.169.254/latest/meta-data/placement/region)"

echo "🔍 IAM Role Check:"
if curl -s -f http://169.254.169.254/latest/meta-data/iam/security-credentials/ > /dev/null; then
    ROLE=$(curl -s http://169.254.169.254/latest/meta-data/iam/security-credentials/)
    echo "   ✅ IAM Role: $ROLE"
    
    # TODO: Fetch the actual credentials for this role using the metadata service
    # TODO: Use the ROLE variable in the endpoint: /latest/meta-data/iam/security-credentials/$ROLE
    
    # TODO: Extract AccessKeyId from the JSON response (show first 20 chars for security)
    # TODO: Extract SecretAccessKey from the JSON response (show first 20 chars for security)  
    # TODO: Extract Expiration time from the JSON response
    
    # TODO: Display the credential information with appropriate labels
    
else
    echo "   ❌ No IAM role attached"
fi

echo "💡 This demonstrates EC2 metadata service access!"
echo "============ EC2 DEMO END ============"
```

**main.py**

```python
import boto3
import time
from botocore.exceptions import ClientError

def load_user_data_script():
    """Load the bash script for EC2 user data"""
    try:
        with open('ec2_demo.sh', 'r') as f:
            return f.read()
    except FileNotFoundError:
        print("❌ Error: ec2_demo.sh file not found")
        return None

def get_latest_amazon_linux_ami():
    """Get the latest Amazon Linux 2 AMI ID for the current region"""
    ssm = boto3.client('ssm')
    try:
        response = ssm.get_parameter(
            Name='/aws/service/ami-amazon-linux-latest/amzn2-ami-hvm-x86_64-gp2'
        )
        return response['Parameter']['Value']
    except Exception as e:
        print(f"❌ Error getting latest AMI: {e}")
        return None


def launch_ec2_demo():
    """Launch EC2 instance with simple bash demo"""
    print("🚀 Launching EC2 instance...")
    
    user_data = load_user_data_script()
    if not user_data:
        return None
    
    ec2 = boto3.client('ec2')
    
    try:
        response = ec2.run_instances(
            ImageId=get_latest_amazon_linux_ami(),
            MinCount=1,
            MaxCount=1,
            InstanceType='t3.micro',
            IamInstanceProfile={'Name': 'learner-metadata-demo'},
            UserData=user_data,
            TagSpecifications=[
                {
                    'ResourceType': 'instance',
                    'Tags': [{'Key': 'Name', 'Value': 'metadata-demo'}]
                }
            ]
        )
        
        instance_id = response['Instances'][0]['InstanceId']
        print(f"✅ Launched instance: {instance_id}")
        return instance_id
        
    except Exception as e:
        print(f"❌ Error launching instance: {e}")
        return None

def wait_for_console_output(instance_id):
    """Wait patiently for console output to appear and parse it correctly"""
    print(f"\n⏳ Waiting for console output from {instance_id}...")
    print("💡 This can take 5-10 minutes - please be patient!")
    
    ec2 = boto3.client('ec2')
    
    # Wait for running state first
    waiter = ec2.get_waiter('instance_running')
    waiter.wait(InstanceIds=[instance_id])
    print("✅ Instance is running")
    
    # Wait longer for console output - up to 15 minutes
    for attempt in range(15):  # Try for up to 15 minutes
        wait_time = 60  # 1 minute intervals
        print(f"⏳ Checking for output (attempt {attempt + 1}/15) - waiting {wait_time}s...")
        time.sleep(wait_time)
        
        try:
            response = ec2.get_console_output(InstanceId=instance_id)
            console_output = response.get('Output', '')
            
            if console_output and 'EC2 DEMO START' in console_output:
                print("\n📋 Actual Demo Results from EC2:")
                print("=" * 60)
                
                # Parse cloud-init logs to extract demo output
                lines = console_output.split('\n')
                demo_lines = []
                in_demo = False
                
                for line in lines:
                    if 'EC2 DEMO START' in line:
                        in_demo = True
                        # Extract message from cloud-init log
                        if 'cloud-init[' in line and ']: ' in line:
                            msg = line.split(']: ', 1)[1]
                            demo_lines.append(msg)
                    elif 'EC2 DEMO END' in line:
                        if 'cloud-init[' in line and ']: ' in line:
                            msg = line.split(']: ', 1)[1]
                            demo_lines.append(msg)
                        break
                    elif in_demo and 'cloud-init[' in line and ']: ' in line:
                        # Extract just the message part after ]: 
                        msg = line.split(']: ', 1)[1]
                        demo_lines.append(msg)
                
                # Display the extracted demo output
                if demo_lines:
                    for demo_line in demo_lines:
                        print(demo_line)
                else:
                    print("Demo markers found but content extraction failed")
                    print("Raw demo section:")
                    for line in lines:
                        if 'DEMO' in line:
                            print(line)
                            
                print("=" * 60)
                return True
                
            elif console_output:
                char_count = len(console_output)
                print(f"   📄 Console output available ({char_count} chars) but demo not found")
                
                # Show some context on later attempts
                if attempt >= 8:
                    print("   Showing lines with 'cloud-init' for debugging:")
                    lines = console_output.split('\n')
                    cloud_init_lines = [line for line in lines if 'cloud-init[' in line]
                    for line in cloud_init_lines[-10:]:  # Last 10 cloud-init lines
                        print(f"   {line}")
            else:
                print("   📭 No console output yet")
                
        except Exception as e:
            print(f"   ❌ Error checking output: {e}")
    
    print("⏰ Timeout waiting for demo output")
    print(f"💡 You can check manually with:")
    print(f"   aws ec2 get-console-output --instance-id {instance_id} --query 'Output' --output text | grep -A 10 -B 2 DEMO")
    return False

def cleanup(instance_id):
    """Terminate the demo instance"""
    if not instance_id:
        return
        
    terminate = input(f"\n🗑️ Terminate instance {instance_id}? (y/N): ").strip().lower()
    if terminate == 'y':
        try:
            ec2 = boto3.client('ec2')
            ec2.terminate_instances(InstanceIds=[instance_id])
            print("✅ Instance terminated")
        except Exception as e:
            print(f"❌ Error: {e}")
    else:
        print(f"⚠️ Instance {instance_id} still running")

def main():
    print("🚀 EC2 IAM Role Credential Discovery Demo")
    print()
    print("💡 This demo shows REAL execution from EC2:")
    print("   • How EC2 instances access the metadata service")
    print("   • Where IAM role credentials are provided") 
    print("   • How boto3 would discover credentials automatically")
    print()
    
    instance_id = launch_ec2_demo()
    
    if instance_id:
        success = wait_for_console_output(instance_id)
        
        if success:
            print("\n🎯 What this demonstrated:")
            print("   ✅ User data script executed successfully on EC2")
            print("   ✅ EC2 metadata service is accessible from instances")
            print("   ✅ Shows actual instance information (ID, AMI, type, region)")
            print("   ✅ Displays temporary IAM role credentials from metadata service")
            print("   💡 boto3 automatically discovers and uses these credentials")
            print("   💡 Credentials are temporary and rotate automatically")
        else:
            print("\n💡 The demo ran, but console output timing varies")
            print("   You can check the results manually with the command shown above")
        
        print("\n🔑 Key Learning Points:")
        print("   • EC2 metadata service URL: http://169.254.169.254/latest/meta-data/")
        print("   • IAM role credentials URL: .../iam/security-credentials/RoleName")  
        print("   • boto3 automatically discovers and uses these credentials")
        print("   • Temporary credentials include AccessKeyId, SecretAccessKey, and SessionToken")
        print("   • Credentials expire and are automatically rotated by AWS")
        
        cleanup(instance_id)

if __name__ == "__main__":
    main()
```

Here is the completed `ec2_demo.sh` with the temporary credentials fetched, parsed, and displayed:

```shell
#!/bin/bash

echo "============ EC2 DEMO START ============"
echo "✅ User data script executing on EC2!"
echo "🔍 Instance Information:"

# Get metadata using curl
echo "   Instance ID: $(curl -s http://169.254.169.254/latest/meta-data/instance-id)"
echo "   AMI ID: $(curl -s http://169.254.169.254/latest/meta-data/ami-id)"  
echo "   Instance Type: $(curl -s http://169.254.169.254/latest/meta-data/instance-type)"
echo "   Region: $(curl -s http://169.254.169.254/latest/meta-data/placement/region)"

echo "🔍 IAM Role Check:"
if curl -s -f http://169.254.169.254/latest/meta-data/iam/security-credentials/ > /dev/null; then
    ROLE=$(curl -s http://169.254.169.254/latest/meta-data/iam/security-credentials/)
    echo "   ✅ IAM Role: $ROLE"
    
    CREDS=$(curl -s http://169.254.169.254/latest/meta-data/iam/security-credentials/$ROLE)
    
    ACCESS_KEY=$(echo "$CREDS" | grep -o '"AccessKeyId" *: *"[^"]*"' | cut -d'"' -f4)
    SECRET_KEY=$(echo "$CREDS" | grep -o '"SecretAccessKey" *: *"[^"]*"' | cut -d'"' -f4)
    EXPIRATION=$(echo "$CREDS" | grep -o '"Expiration" *: *"[^"]*"' | cut -d'"' -f4)
    
    echo "   🔑 Access Key (first 20 chars): ${ACCESS_KEY:0:20}"
    echo "   🔑 Secret Key (first 20 chars): ${SECRET_KEY:0:20}"
    echo "   ⏰ Expires: $EXPIRATION"
    
else
    echo "   ❌ No IAM role attached"
fi

echo "💡 This demonstrates EC2 metadata service access!"
echo "============ EC2 DEMO END ============"
```

`main.py` stays unchanged — it already attaches the `learner-metadata-demo` IAM role via `IamInstanceProfile`, which is what makes real temporary credentials available at the metadata endpoint.

## Validating Metadata Field Retrieval

Nice work! You've successfully explored how to launch instances and parse their console output. Now, let's add some practical validation to ensure the metadata service is working as expected.

Your task is to enhance the `wait_for_console_output()` function in `main.py` to count and validate the metadata fields that were successfully retrieved from the EC2 instance. This kind of validation is useful in real projects to verify that your instance can properly access the metadata service.

You need to add logic that:

* Defines the four expected metadata fields: `"Instance ID"`, `"AMI ID"`, `"Instance Type"`, and `"Region"`
* Loops through the extracted demo output lines to check for each field
* Counts how many fields were found successfully
* Prints a summary line showing the count (like `✅ Retrieved 4/4 metadata fields successfully`)

Look for the TODO comments in the `wait_for_console_output()` function — they will guide you to add this validation logic right after the demo lines are extracted, but before they are displayed to the user. This exercise will give you valuable experience with console output parsing and string manipulation techniques that are essential for monitoring AWS applications.

**main.py**

```python
import boto3
import time
from botocore.exceptions import ClientError

def load_user_data_script():
    """Load the bash script for EC2 user data"""
    try:
        with open('ec2_demo.sh', 'r') as f:
            return f.read()
    except FileNotFoundError:
        print("❌ Error: ec2_demo.sh file not found")
        return None

def get_latest_amazon_linux_ami():
    """Get the latest Amazon Linux 2 AMI ID for the current region"""
    ssm = boto3.client('ssm')
    try:
        response = ssm.get_parameter(
            Name='/aws/service/ami-amazon-linux-latest/amzn2-ami-hvm-x86_64-gp2'
        )
        return response['Parameter']['Value']
    except Exception as e:
        print(f"❌ Error getting latest AMI: {e}")
        return None

def launch_ec2_demo():
    """Launch EC2 instance with simple bash demo"""
    print("🚀 Launching EC2 instance...")
    
    user_data = load_user_data_script()
    if not user_data:
        return None
    
    ec2 = boto3.client('ec2')
    
    try:
        response = ec2.run_instances(
            ImageId=get_latest_amazon_linux_ami(),
            MinCount=1,
            MaxCount=1,
            InstanceType='t3.micro',
            UserData=user_data,
            TagSpecifications=[
                {
                    'ResourceType': 'instance',
                    'Tags': [{'Key': 'Name', 'Value': 'metadata-demo'}]
                }
            ]
        )
        
        instance_id = response['Instances'][0]['InstanceId']
        print(f"✅ Launched instance: {instance_id}")
        return instance_id
        
    except Exception as e:
        print(f"❌ Error launching instance: {e}")
        return None

def wait_for_console_output(instance_id):
    """Wait patiently for console output to appear and parse it correctly"""
    print(f"\n⏳ Waiting for console output from {instance_id}...")
    print("💡 This can take 5-10 minutes - please be patient!")
    
    ec2 = boto3.client('ec2')
    
    # Wait for running state first
    waiter = ec2.get_waiter('instance_running')
    waiter.wait(InstanceIds=[instance_id])
    print("✅ Instance is running")
    
    # Wait longer for console output - up to 15 minutes
    for attempt in range(15):  # Try for up to 15 minutes
        wait_time = 60  # 1 minute intervals
        print(f"⏳ Checking for output (attempt {attempt + 1}/15) - waiting {wait_time}s...")
        time.sleep(wait_time)
        
        try:
            response = ec2.get_console_output(InstanceId=instance_id)
            console_output = response.get('Output', '')
            
            if console_output and 'EC2 DEMO START' in console_output:
                print("\n📋 Actual Demo Results from EC2:")
                print("=" * 60)
                
                # Parse cloud-init logs to extract demo output
                lines = console_output.split('\n')
                demo_lines = []
                in_demo = False
                
                for line in lines:
                    if 'EC2 DEMO START' in line:
                        in_demo = True
                        # Extract message from cloud-init log
                        if 'cloud-init[' in line and ']: ' in line:
                            msg = line.split(']: ', 1)[1]
                            demo_lines.append(msg)
                    elif 'EC2 DEMO END' in line:
                        if 'cloud-init[' in line and ']: ' in line:
                            msg = line.split(']: ', 1)[1]
                            demo_lines.append(msg)
                        break
                    elif in_demo and 'cloud-init[' in line and ']: ' in line:
                        # Extract just the message part after ]: 
                        msg = line.split(']: ', 1)[1]
                        demo_lines.append(msg)
                
                # TODO: Create a list of expected metadata fields: "Instance ID", "AMI ID", "Instance Type", "Region"
                # TODO: Initialize a counter to track how many fields were found
                # TODO: Loop through each expected field and check if it appears in any demo line
                # TODO: For each field found, increment the counter
                # TODO: Print a summary line showing "Retrieved X/4 metadata fields successfully"
                # TODO: Add an empty print() for spacing
                
                # Display the extracted demo output
                if demo_lines:
                    for demo_line in demo_lines:
                        print(demo_line)
                else:
                    print("Demo markers found but content extraction failed")
                    print("Raw demo section:")
                    for line in lines:
                        if 'DEMO' in line:
                            print(line)
                            
                print("=" * 60)
                return True
                
            elif console_output:
                char_count = len(console_output)
                print(f"   📄 Console output available ({char_count} chars) but demo not found")
                
                # Show some context on later attempts
                if attempt >= 8:
                    print("   Showing lines with 'cloud-init' for debugging:")
                    lines = console_output.split('\n')
                    cloud_init_lines = [line for line in lines if 'cloud-init[' in line]
                    for line in cloud_init_lines[-10:]:  # Last 10 cloud-init lines
                        print(f"   {line}")
            else:
                print("   📭 No console output yet")
                
        except Exception as e:
            print(f"   ❌ Error checking output: {e}")
    
    print("⏰ Timeout waiting for demo output")
    print(f"💡 You can check manually with:")
    print(f"   aws ec2 get-console-output --instance-id {instance_id} --query 'Output' --output text | grep -A 10 -B 2 DEMO")
    return False

def cleanup(instance_id):
    """Terminate the demo instance"""
    if not instance_id:
        return
        
    terminate = input(f"\n🗑️ Terminate instance {instance_id}? (y/N): ").strip().lower()
    if terminate == 'y':
        try:
            ec2 = boto3.client('ec2')
            ec2.terminate_instances(InstanceIds=[instance_id])
            print("✅ Instance terminated")
        except Exception as e:
            print(f"❌ Error: {e}")
    else:
        print(f"⚠️ Instance {instance_id} still running")

def main():
    print("🚀 EC2 IAM Role Credential Discovery Demo")
    print()
    print("💡 This demo shows REAL execution from EC2:")
    print("   • How EC2 instances access the metadata service")
    print("   • Where IAM role credentials are provided") 
    print("   • How boto3 would discover credentials automatically")
    print()
    
    instance_id = launch_ec2_demo()
    
    if instance_id:
        success = wait_for_console_output(instance_id)
        
        if success:
            print("\n🎯 What this demonstrated:")
            print("   ✅ User data script executed successfully on EC2")
            print("   ✅ EC2 metadata service is accessible from instances")
            print("   ✅ Shows actual instance information (ID, AMI, type, region)")
            print("   ✅ Demonstrates where IAM role credentials would appear")
            print("   💡 Without IAM role: No AWS API access")
            print("   💡 With IAM role: boto3 automatically gets credentials from metadata service")
        else:
            print("\n💡 The demo ran, but console output timing varies")
            print("   You can check the results manually with the command shown above")
        
        print("\n🔑 Key Learning Points:")
        print("   • EC2 metadata service URL: http://169.254.169.254/latest/meta-data/")
        print("   • IAM role credentials URL: .../iam/security-credentials/RoleName")  
        print("   • boto3 automatically discovers and uses these credentials")
        print("   • No IAM role = NoCredentialsError when calling AWS APIs")
        
        cleanup(instance_id)

if __name__ == "__main__":
    main()
```

**ec2_demo.sh**

```shell
#!/bin/bash

echo "============ EC2 DEMO START ============"
echo "✅ User data script executing on EC2!"
echo "🔍 Instance Information:"

# Get metadata using curl
echo "   Instance ID: $(curl -s http://169.254.169.254/latest/meta-data/instance-id)"
echo "   AMI ID: $(curl -s http://169.254.169.254/latest/meta-data/ami-id)"  
echo "   Instance Type: $(curl -s http://169.254.169.254/latest/meta-data/instance-type)"
echo "   Region: $(curl -s http://169.254.169.254/latest/meta-data/placement/region)"

echo "🔍 IAM Role Check:"
if curl -s -f http://169.254.169.254/latest/meta-data/iam/security-credentials/ > /dev/null; then
    ROLE=$(curl -s http://169.254.169.254/latest/meta-data/iam/security-credentials/)
    echo "   ✅ IAM Role: $ROLE"
else
    echo "   ❌ No IAM role attached"
fi

echo "💡 This demonstrates EC2 metadata service access!"
echo "============ EC2 DEMO END ============"
```

Here is the completed `wait_for_console_output()` with the validation logic added right after the demo lines are extracted:

```python
def wait_for_console_output(instance_id):
    """Wait patiently for console output to appear and parse it correctly"""
    print(f"\n⏳ Waiting for console output from {instance_id}...")
    print("💡 This can take 5-10 minutes - please be patient!")
    
    ec2 = boto3.client('ec2')
    
    # Wait for running state first
    waiter = ec2.get_waiter('instance_running')
    waiter.wait(InstanceIds=[instance_id])
    print("✅ Instance is running")
    
    # Wait longer for console output - up to 15 minutes
    for attempt in range(15):  # Try for up to 15 minutes
        wait_time = 60  # 1 minute intervals
        print(f"⏳ Checking for output (attempt {attempt + 1}/15) - waiting {wait_time}s...")
        time.sleep(wait_time)
        
        try:
            response = ec2.get_console_output(InstanceId=instance_id)
            console_output = response.get('Output', '')
            
            if console_output and 'EC2 DEMO START' in console_output:
                print("\n📋 Actual Demo Results from EC2:")
                print("=" * 60)
                
                # Parse cloud-init logs to extract demo output
                lines = console_output.split('\n')
                demo_lines = []
                in_demo = False
                
                for line in lines:
                    if 'EC2 DEMO START' in line:
                        in_demo = True
                        # Extract message from cloud-init log
                        if 'cloud-init[' in line and ']: ' in line:
                            msg = line.split(']: ', 1)[1]
                            demo_lines.append(msg)
                    elif 'EC2 DEMO END' in line:
                        if 'cloud-init[' in line and ']: ' in line:
                            msg = line.split(']: ', 1)[1]
                            demo_lines.append(msg)
                        break
                    elif in_demo and 'cloud-init[' in line and ']: ' in line:
                        # Extract just the message part after ]: 
                        msg = line.split(']: ', 1)[1]
                        demo_lines.append(msg)
                
                expected_fields = ["Instance ID", "AMI ID", "Instance Type", "Region"]
                found_count = 0
                for field in expected_fields:
                    if any(field in demo_line for demo_line in demo_lines):
                        found_count += 1
                print(f"✅ Retrieved {found_count}/{len(expected_fields)} metadata fields successfully")
                print()
                
                # Display the extracted demo output
                if demo_lines:
                    for demo_line in demo_lines:
                        print(demo_line)
                else:
                    print("Demo markers found but content extraction failed")
                    print("Raw demo section:")
                    for line in lines:
                        if 'DEMO' in line:
                            print(line)
                            
                print("=" * 60)
                return True
                
            elif console_output:
                char_count = len(console_output)
                print(f"   📄 Console output available ({char_count} chars) but demo not found")
                
                # Show some context on later attempts
                if attempt >= 8:
                    print("   Showing lines with 'cloud-init' for debugging:")
                    lines = console_output.split('\n')
                    cloud_init_lines = [line for line in lines if 'cloud-init[' in line]
                    for line in cloud_init_lines[-10:]:  # Last 10 cloud-init lines
                        print(f"   {line}")
            else:
                print("   📭 No console output yet")
                
        except Exception as e:
            print(f"   ❌ Error checking output: {e}")
    
    print("⏰ Timeout waiting for demo output")
    print(f"💡 You can check manually with:")
    print(f"   aws ec2 get-console-output --instance-id {instance_id} --query 'Output' --output text | grep -A 10 -B 2 DEMO")
    return False
```

## Smart Instance State Management

Excellent work on validating metadata field retrieval! You've built solid skills in console output parsing and validation techniques. Now, let's add an important layer of reliability by implementing proper instance state management before cleanup operations.

Your task is to create smart state checking that ensures cleanup operations are performed only when appropriate. In production AWS applications, you should always verify resource states before attempting operations on them.

You need to implement two key components:

* Create a `check_instance_state()` function that uses the `describe_instances` API to get the current instance state
* Enhance the `cleanup()` function to handle different instance states appropriately
* Add conditional logic for states like running, pending, stopped, and terminated
* Display helpful messages for each state scenario

This exercise will teach you essential instance lifecycle management and defensive programming practices that prevent errors in real AWS applications.

Once you complete this change, you can run the Python script to launch your EC2 instance and see your new availability zone information appear alongside the other instance details in the console output.

**main.py**

```python
import boto3
import time
from botocore.exceptions import ClientError

def check_instance_state(instance_id):
    """Check the current state of an EC2 instance"""
    try:
        # TODO: Create an EC2 client using boto3
        # TODO: Use describe_instances() API call with the instance_id
        # TODO: Extract the state from response['Reservations'][0]['Instances'][0]['State']['Name']
        # TODO: Print the current state with an appropriate message
        # TODO: Return the state string
        pass
    except Exception as e:
        print(f"❌ Error checking instance state: {e}")
        return None

def load_user_data_script():
    """Load the bash script for EC2 user data"""
    try:
        with open('ec2_demo.sh', 'r') as f:
            return f.read()
    except FileNotFoundError:
        print("❌ Error: ec2_demo.sh file not found")
        return None

def get_latest_amazon_linux_ami():
    """Get the latest Amazon Linux 2 AMI ID for the current region"""
    ssm = boto3.client('ssm')
    try:
        response = ssm.get_parameter(
            Name='/aws/service/ami-amazon-linux-latest/amzn2-ami-hvm-x86_64-gp2'
        )
        return response['Parameter']['Value']
    except Exception as e:
        print(f"❌ Error getting latest AMI: {e}")
        return None

def launch_ec2_demo():
    """Launch EC2 instance with simple bash demo"""
    print("🚀 Launching EC2 instance...")

    user_data = load_user_data_script()
    if not user_data:
        return None

    ec2 = boto3.client('ec2')

    try:
        response = ec2.run_instances(
            ImageId=get_latest_amazon_linux_ami(),
            MinCount=1,
            MaxCount=1,
            InstanceType='t3.micro',
            UserData=user_data,
            TagSpecifications=[
                {
                    'ResourceType': 'instance',
                    'Tags': [{'Key': 'Name', 'Value': 'metadata-demo'}]
                }
            ]
        )

        instance_id = response['Instances'][0]['InstanceId']
        print(f"✅ Launched instance: {instance_id}")
        return instance_id

    except Exception as e:
        print(f"❌ Error launching instance: {e}")
        return None

def wait_for_console_output(instance_id):
    """Wait patiently for console output to appear and parse it correctly"""
    print(f"\n⏳ Waiting for console output from {instance_id}...")
    print("💡 This can take 5-10 minutes - please be patient!")

    ec2 = boto3.client('ec2')

    # Wait for running state first
    waiter = ec2.get_waiter('instance_running')
    waiter.wait(InstanceIds=[instance_id])
    print("✅ Instance is running")

    # Wait longer for console output - up to 15 minutes
    for attempt in range(15):  # Try for up to 15 minutes
        wait_time = 60  # 1 minute intervals
        print(f"⏳ Checking for output (attempt {attempt + 1}/15) - waiting {wait_time}s...")
        time.sleep(wait_time)

        try:
            response = ec2.get_console_output(InstanceId=instance_id)
            console_output = response.get('Output', '')

            if console_output and 'EC2 DEMO START' in console_output:
                print("\n📋 Actual Demo Results from EC2:")
                print("=" * 60)

                # Parse cloud-init logs to extract demo output
                lines = console_output.split('\n')
                demo_lines = []
                in_demo = False

                for line in lines:
                    if 'EC2 DEMO START' in line:
                        in_demo = True
                        # Extract message from cloud-init log
                        if 'cloud-init[' in line and ']: ' in line:
                            msg = line.split(']: ', 1)[1]
                            demo_lines.append(msg)
                    elif 'EC2 DEMO END' in line:
                        if 'cloud-init[' in line and ']: ' in line:
                            msg = line.split(']: ', 1)[1]
                            demo_lines.append(msg)
                        break
                    elif in_demo and 'cloud-init[' in line and ']: ' in line:
                        # Extract just the message part after ]:
                        msg = line.split(']: ', 1)[1]
                        demo_lines.append(msg)

                # Display the extracted demo output
                if demo_lines:
                    for demo_line in demo_lines:
                        print(demo_line)
                else:
                    print("Demo markers found but content extraction failed")
                    print("Raw demo section:")
                    for line in lines:
                        if 'DEMO' in line:
                            print(line)

                print("=" * 60)
                return True

            elif console_output:
                char_count = len(console_output)
                print(f"   📄 Console output available ({char_count} chars) but demo not found")

                # Show some context on later attempts
                if attempt >= 8:
                    print("   Showing lines with 'cloud-init' for debugging:")
                    lines = console_output.split('\n')
                    cloud_init_lines = [line for line in lines if 'cloud-init[' in line]
                    for line in cloud_init_lines[-10:]:  # Last 10 cloud-init lines
                        print(f"   {line}")
            else:
                print("   📭 No console output yet")

        except Exception as e:
            print(f"   ❌ Error checking output: {e}")

    print("⏰ Timeout waiting for demo output")
    print(f"💡 You can check manually with:")
    print(f"   aws ec2 get-console-output --instance-id {instance_id} --query 'Output' --output text | grep -A 10 -B 2 DEMO")
    return False

def cleanup(instance_id):
    """Terminate the demo instance"""
    if not instance_id:
        return

    # TODO: Call check_instance_state() to get the current state

    # TODO: Check if state is 'running' and only then show termination prompt
    terminate = input(f"\n🗑️ Terminate instance {instance_id}? (y/N): ").strip().lower()
    if terminate == 'y':
        try:
            ec2 = boto3.client('ec2')
            ec2.terminate_instances(InstanceIds=[instance_id])
            print("✅ Instance terminated")
        except Exception as e:
            print(f"❌ Error: {e}")
    else:
        print(f"⚠️ Instance {instance_id} still running")
    # TODO: Handle 'pending' state with message "Instance is still starting up - no action needed"
    # TODO: Handle 'stopped' state with message "Instance is already stopped - no action needed"
    # TODO: Handle 'terminated'/'terminating' states with message "Instance is already being terminated"
    # TODO: Handle other states with a generic message showing the current state

def main():
    print("🚀 EC2 IAM Role Credential Discovery Demo")
    print()
    print("💡 This demo shows REAL execution from EC2:")
    print("   • How EC2 instances access the metadata service")
    print("   • Where IAM role credentials are provided")
    print("   • How boto3 would discover credentials automatically")
    print()

    instance_id = launch_ec2_demo()

    if instance_id:
        success = wait_for_console_output(instance_id)

        if success:
            print("\n🎯 What this demonstrated:")
            print("   ✅ User data script executed successfully on EC2")
            print("   ✅ EC2 metadata service is accessible from instances")
            print("   ✅ Shows actual instance information (ID, AMI, type, region)")
            print("   ✅ Demonstrates where IAM role credentials would appear")
            print("   💡 Without IAM role: No AWS API access")
            print("   💡 With IAM role: boto3 automatically gets credentials from metadata service")
        else:
            print("\n💡 The demo ran, but console output timing varies")
            print("   You can check the results manually with the command shown above")

        print("\n🔑 Key Learning Points:")
        print("   • EC2 metadata service URL: http://169.254.169.254/latest/meta-data/")
        print("   • IAM role credentials URL: .../iam/security-credentials/RoleName")
        print("   • boto3 automatically discovers and uses these credentials")
        print("   • No IAM role = NoCredentialsError when calling AWS APIs")

        cleanup(instance_id)

if __name__ == "__main__":
    main()
```

**ec2_demo.sh**

```shell
#!/bin/bash

echo "============ EC2 DEMO START ============"
echo "✅ User data script executing on EC2!"
echo "🔍 Instance Information:"

# Get metadata using curl
echo "   Instance ID: $(curl -s http://169.254.169.254/latest/meta-data/instance-id)"
echo "   AMI ID: $(curl -s http://169.254.169.254/latest/meta-data/ami-id)"
echo "   Instance Type: $(curl -s http://169.254.169.254/latest/meta-data/instance-type)"
echo "   Region: $(curl -s http://169.254.169.254/latest/meta-data/placement/region)"

echo "🔍 IAM Role Check:"
if curl -s -f http://169.254.169.254/latest/meta-data/iam/security-credentials/ > /dev/null; then
    ROLE=$(curl -s http://169.254.169.254/latest/meta-data/iam/security-credentials/)
    echo "   ✅ IAM Role: $ROLE"
else
    echo "   ❌ No IAM role attached"
fi

echo "💡 This demonstrates EC2 metadata service access!"
echo "============ EC2 DEMO END ============"
```

Here are the completed `check_instance_state()` and `cleanup()` functions:

```python
def check_instance_state(instance_id):
    """Check the current state of an EC2 instance"""
    try:
        ec2 = boto3.client('ec2')
        response = ec2.describe_instances(InstanceIds=[instance_id])
        state = response['Reservations'][0]['Instances'][0]['State']['Name']
        print(f"📊 Current instance state: {state}")
        return state
    except Exception as e:
        print(f"❌ Error checking instance state: {e}")
        return None

def cleanup(instance_id):
    """Terminate the demo instance"""
    if not instance_id:
        return

    state = check_instance_state(instance_id)

    if state == 'running':
        terminate = input(f"\n🗑️ Terminate instance {instance_id}? (y/N): ").strip().lower()
        if terminate == 'y':
            try:
                ec2 = boto3.client('ec2')
                ec2.terminate_instances(InstanceIds=[instance_id])
                print("✅ Instance terminated")
            except Exception as e:
                print(f"❌ Error: {e}")
        else:
            print(f"⚠️ Instance {instance_id} still running")
    elif state == 'pending':
        print("💡 Instance is still starting up - no action needed")
    elif state == 'stopped':
        print("💡 Instance is already stopped - no action needed")
    elif state in ('terminated', 'terminating'):
        print("💡 Instance is already being terminated")
    else:
        print(f"💡 Instance is in state '{state}' - no action taken")
```